In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
gdp = spark.table("scottish_housing_ws.silver.scottish_gdp")
hpi = spark.table("scottish_housing_ws.silver.uk_hpi")

In [0]:
gdp.printSchema()
gdp.show(5)

In [0]:
gdp.select("measure_type", "units").distinct().orderBy("measure_type").show()

In [0]:
# Pivot GDP measure types onto one row per quarter 
# Index (2022=100) is the primary series -- level of economic output
# q-on-q is quarter on quarter growth -- short term momentum
# q-on-q year ago is annual growth -- smooths out seasonal effects
# 4q-on-4q is four quarter rolling growth -- best for trend analysis

gdp_pivoted = (
    gdp
    .groupBy("report_date", "year_quarter")
    .pivot("measure_type", ["Index", "q-on-q", "q-on-q year ago", "4q-on-4q"])
    .agg(F.first("value"))
    .withColumnRenamed("Index", "gdp_index")
    .withColumnRenamed("q-on-q", "gdp_qoq_growth")
    .withColumnRenamed("q-on-q year ago", "gdp_qoq_year_ago")
    .withColumnRenamed("4q-on-4q", "gdp_4q_rolling_growth")
    .orderBy("year_quarter")
)

In [0]:
hpi_quarterly = (
    hpi
    .filter(F.col("area_code") == "S92000003")
    .withColumn("year", F.col("year_month").substr(1, 4).cast("int"))
    .withColumn("month", F.col("year_month").substr(6, 2).cast("int"))
    .withColumn(
        "quarter",
        F.when(F.col("month").between(1, 3), "Q1")
        .when(F.col("month").between(4, 6), "Q2")
        .when(F.col("month").between(7, 9), "Q3")
        .otherwise("Q4")
    )
    .withColumn(
        "year_quarter",
        F.concat(F.col("year").cast("string"), F.lit("-"), F.col("quarter"))
    )
    .groupBy("year_quarter")
    .agg(
        F.round(F.avg("average_price"), 0).alias("avg_house_price"),
        F.sum("sales_volume").alias("total_sales_volume")
    )
)

In [0]:
gold = (
    gdp_pivoted
    .join(hpi_quarterly, on="year_quarter", how="inner")
    .select(
        "report_date",
        "year_quarter",
        "gdp_index",
        "gdp_qoq_growth",
        "gdp_qoq_year_ago",
        "gdp_4q_rolling_growth",
        "avg_house_price",
        "total_sales_volume"
    )
    .orderBy("year_quarter")
)

In [0]:
print(f"Row count: {gold.count()}")
gold.orderBy("year_quarter").show(5)
gold.orderBy(F.col("year_quarter").desc()).show(5)

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.gdp_vs_house_price")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_gdp_vs_house_price/")
)

print("Gold 14 complete.")

In [0]:
# GDP vs House Price
# Sources: silver.scottish_gdp, silver.uk_hpi
# Grain: quarterly, Scotland national only
# Join key: year_quarter
#
# Scottish GDP is onshore only -- excludes North Sea oil and gas.
# GDP measure types: Index, q-on-q, q-on-q year ago, 4q-on-4q
# We use the Index measure (2022=100) as the primary GDP series
# and include q-on-q growth for context.
#
# HPI is aggregated from monthly to quarterly by averaging
# average_price across the three months in each quarter.
#
# Coverage overlap: GDP 1998 Q1 to 2025 Q4, HPI from 1968
# Effective range after join: 1998 Q1 to 2025 Q4